# GEC Pipeline - Training and Inference

This notebook walks through the complete GEC (Grammatical Error Correction) pipeline:
1. Data preparation and feature extraction
2. Training the edit tagger model
3. Running inference on new text

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

project_root = Path.cwd().parents[3]
sys.path.insert(0, str(project_root))

print("Setup complete!")

Setup complete!


## Part 1: Data Preparation

In [2]:
from src.services.gec.config import (
    TRAIN_SENT_PATH,
    TRAIN_COR_PATH,
    NOPNX_TRAIN_OUTPUT,
    PNX_TRAIN_OUTPUT,
    LABEL2ID_PATH,
    ID2LABEL_PATH,
    CHECKPOINT_PATH,
)

print("Checking data files...")
print(f"Training sentences: {TRAIN_SENT_PATH.exists()}")
print(f"Training corrections: {TRAIN_COR_PATH.exists()}")
print(f"Checkpoint (processed data): {CHECKPOINT_PATH.exists()}")

Checking data files...
Training sentences: True
Training corrections: True
Checkpoint (processed data): True


In [3]:
import json

from src.services.gec.features.build_train import build_train

if CHECKPOINT_PATH.exists():
    print("Loading existing processed data...")
    with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
        num_lines = sum(1 for _ in f)
    print(f"✓ Found {num_lines} training examples")
    
    with open(LABEL2ID_PATH, 'r', encoding='utf-8') as f:
        label2id = json.load(f)
    print(f"✓ Label vocabulary: {len(label2id)} labels")
else:
    print("⚠️  No processed data found. Run build_train() first.")
    build_train()

/home/somia/baligh/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading existing processed data...
✓ Found 501 training examples
✓ Label vocabulary: 362 labels


In [4]:
print("Sample training example:")
with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
    first_line = f.readline()
    example = json.loads(first_line.strip())
    print(f"Subwords (first 15): {example['subwords'][:15]}")
    print(f"Labels (first 15): {example['labels'][:15]}")
    print(f"Total subwords: {len(example['subwords'])}")
    print(f"Total labels: {len(example['labels'])}")
    print(f"Length match: {len(example['subwords']) == len(example['labels'])}")

Sample training example:
Subwords (first 15): ['الى', 'التعليق', 'رقم', '2', ' ', 'اك', '##يد', 'ان', 'لحكام', 'العرب', 'والمسلمين', 'مسؤولية', 'يتمثل', 'اد', '##ناها']
Labels (first 15): ['R_[إ]K2', 'K7', 'K3', 'K', 'R_[:]', 'R_[أ]K', 'K2', 'R_[أ]K', 'I_[ل]K5', 'K5', 'K9', 'K7', 'K5', 'R_[أ]K', 'K4']
Total subwords: 56
Total labels: 56
Length match: True


## Part 2: Model Training

In [5]:
from transformers import AutoTokenizer
from src.services.gec.training.datasets import GECTrainingDataset
from src.services.gec.training.trainer import build_trainer

MODEL_CHECKPOINT = "aubmindlab/bert-base-arabertv02"
OUTPUT_DIR = Path("./gec_models/edit_tagger_v1")
NUM_EPOCHS = 3
BATCH_SIZE = 8
LEARNING_RATE = 3e-5
MAX_LENGTH = 256

print(f"Model: {MODEL_CHECKPOINT}")
print(f"Output: {OUTPUT_DIR}")
print(f"Epochs: {NUM_EPOCHS}, Batch: {BATCH_SIZE}, Max length: {MAX_LENGTH}")

Model: aubmindlab/bert-base-arabertv02
Output: gec_models/edit_tagger_v1
Epochs: 3, Batch: 8, Max length: 256


In [6]:
with open(LABEL2ID_PATH, 'r', encoding='utf-8') as f:
    label2id = json.load(f)

with open(ID2LABEL_PATH, 'r', encoding='utf-8') as f:
    id2label = json.load(f)

print(f"Labels: {len(label2id)}")
print(f"\nMost common labels (first 30):")
for i, (label, idx) in enumerate(list(label2id.items())[:30]):
    print(f"  {idx}: {label}")

Labels: 362

Most common labels (first 30):
  0: [PAD]
  1: [UNK_EDIT]
  2: D
  3: D*
  4: D*K
  5: D*K*
  6: D*KD
  7: D*KD*
  8: D*KD*R_[ن]
  9: D*KDR_[ا]
  10: D*KR_[ا]
  11: D*KR_[ن]
  12: D*KR_[نعم*]
  13: D*R_[!]
  14: D*R_["]
  15: D*R_[(]
  16: D*R_[)]
  17: D*R_[-]
  18: D*R_[.]
  19: D*R_[:]
  20: D*R_[،]
  21: D*R_[؛]
  22: D*R_[؟]
  23: D*R_[أ]
  24: D*R_[أ]K
  25: D*R_[أمن*]
  26: D*R_[أن*]
  27: D*R_[أهل*]
  28: D*R_[أو*]
  29: D*R_[إ]


In [7]:
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
print(f"✓ Vocab size: {tokenizer.vocab_size}")
print(f"✓ PAD token: {tokenizer.pad_token}")
print(f"✓ PAD token ID: {tokenizer.pad_token_id}")

Loading tokenizer...


✓ Vocab size: 64000
✓ PAD token: [PAD]
✓ PAD token ID: 0


In [8]:
# print("Loading training dataset...")
# train_dataset = GECTrainingDataset(
#     jsonl_path=CHECKPOINT_PATH,
#     tokenizer=tokenizer,
#     label2id=label2id,
#     max_length=MAX_LENGTH,
# )
# print(f"✓ Training examples: {len(train_dataset)}")

# sample = train_dataset[0]
# print(f"\nSample item structure:")
# print(f"  input_ids length: {len(sample['input_ids'])}")
# print(f"  attention_mask length: {len(sample['attention_mask'])}")
# print(f"  labels length: {len(sample['labels'])}")
# print(f"  Lengths match: {len(sample['input_ids']) == len(sample['labels'])}")

In [9]:
# from src.services.gec.training.model import create_model
# print("Initializing model...")
# model = create_model(
#     checkpoint=MODEL_CHECKPOINT,
#     label2id=label2id,
# )
# print(f"✓ Model initialized with {len(label2id)} output labels")

In [10]:
# print("Building trainer...")
# trainer = build_trainer(
#     model=model,
#     tokenizer= tokenizer,
#     train_dataset=train_dataset,
#     eval_dataset=None,
#     output_dir=OUTPUT_DIR,
#     num_train_epochs=NUM_EPOCHS,
#     learning_rate=LEARNING_RATE,
#     fp16=False,
#     label2id_path=LABEL2ID_PATH,
# )
# print("✓ Trainer ready!")

In [11]:
# print("\n🚀 Starting training...")
# print("="*60)

# trainer.train()

# print("\n" + "="*60)    
# print("✓ Training complete!")

In [12]:
with open(LABEL2ID_PATH, 'r', encoding='utf-8') as f:
    label2id = json.load(f)

with open(ID2LABEL_PATH, 'r', encoding='utf-8') as f:
    id2label = json.load(f)

print(f"✓ Labels (star-compressed): {len(label2id)} unique labels")
print(f"\nLabel vocabulary:")
for i, (label, idx) in enumerate(list(label2id.items())[:40]):
    print(f"  {idx}: {label}")
if len(label2id) > 40:
    print(f"  ... and {len(label2id) - 40} more")

✓ Labels (star-compressed): 362 unique labels

Label vocabulary:
  0: [PAD]
  1: [UNK_EDIT]
  2: D
  3: D*
  4: D*K
  5: D*K*
  6: D*KD
  7: D*KD*
  8: D*KD*R_[ن]
  9: D*KDR_[ا]
  10: D*KR_[ا]
  11: D*KR_[ن]
  12: D*KR_[نعم*]
  13: D*R_[!]
  14: D*R_["]
  15: D*R_[(]
  16: D*R_[)]
  17: D*R_[-]
  18: D*R_[.]
  19: D*R_[:]
  20: D*R_[،]
  21: D*R_[؛]
  22: D*R_[؟]
  23: D*R_[أ]
  24: D*R_[أ]K
  25: D*R_[أمن*]
  26: D*R_[أن*]
  27: D*R_[أهل*]
  28: D*R_[أو*]
  29: D*R_[إ]
  30: D*R_[إلى*]
  31: D*R_[إن*]
  32: D*R_[ا]
  33: D*R_[ب]
  34: D*R_[بعد*]
  35: D*R_[به*]
  36: D*R_[ت]
  37: D*R_[ح]
  38: D*R_[حتى*]
  39: D*R_[د]
  ... and 322 more


## Part 3: Inference

In [13]:
from transformers import AutoModelForTokenClassification
from src.services.gec.modules.edit_tagger.inference import GECInferencePipeline
from src.services.gec.utils.string_utils import Tokenizer


best_model_path = Path("./gec_models/edit_tagger_v1/checkpoint-96")
if not best_model_path.exists():
    print("⚠️  Model not found. Please train first.")
else:
    print(f"Loading model from {best_model_path}...")
    inference_tokenizer = AutoTokenizer.from_pretrained(best_model_path)
    inference_model = AutoModelForTokenClassification.from_pretrained(best_model_path)
    print(f"✓ Model loaded ({inference_model.config.num_labels} labels)")

Loading model from gec_models/edit_tagger_v1/checkpoint-96...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2293.83it/s]

✓ Model loaded (362 labels)


In [14]:
class LabelVocab:
    def __init__(self, id2label):
        self.id2label = id2label

class DummyRewriter:
    pass

if best_model_path.exists():
    custom_tokenizer = Tokenizer()
    label_vocab = LabelVocab(id2label)
    rewriter = DummyRewriter()

    pipeline = GECInferencePipeline(
        model=inference_model,
        tokenizer=custom_tokenizer,
        label_vocab=label_vocab,
    )
    print("✓ Inference pipeline ready!")

✓ Inference pipeline ready!


In [15]:
pipeline.predict("هل تعيسم نون")

AttributeError: 'TokenClassifierOutput' object has no attribute 'argmax'

In [ ]:
print("Sample training example:")
with open(CHECKPOINT_PATH, 'r', encoding='utf-8') as f:
    first_line = f.readline()
    example = json.loads(first_line.strip())
    print(f"Subwords (first 15): {example['subwords'][:15]}")
    print(f"Labels - count version (first 15): {example.get('labels', [])[:15]}")
    print(f"Labels - star version (first 15): {example.get('labels_star', [])[:15]}")
    print(f"\nUsing labels_star for training: {len(example.get('labels_star', []))} labels")
    print(f"Total subwords: {len(example['subwords'])}")
    print(f"Length match: {len(example['subwords']) == len(example.get('labels_star', []))}")

Sample training example:
Subwords (first 15): ['الى', 'التعليق', 'رقم', '2', ' ', 'اك', '##يد', 'ان', 'لحكام', 'العرب', 'والمسلمين', 'مسؤولية', 'يتمثل', 'اد', '##ناها']
Labels - count version (first 15): ['R_[إ]K2', 'K7', 'K3', 'K', 'R_[:]', 'R_[أ]K', 'K2', 'R_[أ]K', 'I_[ل]K5', 'K5', 'K9', 'K7', 'K5', 'R_[أ]K', 'K4']
Labels - star version (first 15): ['R_[إ]K*', 'K*', 'K*', 'K', 'R_[:]', 'R_[أ]K', 'K*', 'R_[أ]K', 'I_[ل]K*', 'K*', 'K*', 'K*', 'K*', 'R_[أ]K', 'K*']

Using labels_star for training: 56 labels
Total subwords: 56
Length match: True
